In [ ]:
# 0. setup: ISI REPO_URL dan (bila repo privat) token akses
!pip install -q xgboost pytest tabulate
import os, sys, shutil, importlib, json, time
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/<pengguna>/<nama-repo>.git"      # <- isi
REPO = "/content/kp2-fog-pose"
if not os.path.isdir(REPO):
    !git clone -q --depth 1 {REPO_URL} {REPO}
!pip install -q -r {REPO}/requirements.txt

DRIVE = Path("/content/drive/MyDrive/KP-TA MCI")                  # folder kerja pribadi (JANGAN dibagikan)
FIGSHARE_DIR = DRIVE / "02. Figshare"
W2, W4 = DRIVE / "05. Week 2", DRIVE / "07. Week 4"
HASIL = W4 / "hasil"; HASIL.mkdir(parents=True, exist_ok=True)

LOCAL = Path("/content/cache")                                    # salin cache ke disk lokal supaya baca cepat
for nm in ("fitur_li_v2", "pose_grid32"):
    if not (LOCAL / nm).exists():
        shutil.copytree(W2 / nm, LOCAL / nm)
FITUR_DIR, POSE_GRID_DIR = LOCAL / "fitur_li_v2", LOCAL / "pose_grid32"

Path(REPO, "config", "paths.local.yaml").write_text(f'figshare: "{FIGSHARE_DIR}"\n', encoding="utf-8")   # tidak ikut git
os.environ.update(FOG_FITUR_DIR=str(FITUR_DIR), FOG_POSE_DIR=str(POSE_GRID_DIR),
                  KP_BASE=str(DRIVE), KP_KODE=f"{REPO}/models")    # runner memakai models/ dari repo
if REPO not in sys.path: sys.path.insert(0, REPO)

from fogbench import data as D, splits, schema
from fogbench.threshold import pick_threshold
from fogbench.metrics import sample_metrics
from fogbench.evaluate import protocol
import torch
recs = D.load("figshare")
print("Rekaman:", len(recs), "| pasien:", len({r.subject for r in recs}))
print("fitur:", len(list(FITUR_DIR.glob("PDFE*.npz"))), "| pose:", len(list(POSE_GRID_DIR.glob("*.npz"))))
print("GPU:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# 1. uji repo: uji kebocoran starter, kesetaraan satu_fold vs run_loso, dan fogbench/ tidak berubah
!cd {REPO} && python -m pytest tests -q 2>&1 | tail -3
!cd {REPO} && python scripts/selftest.py 2>&1 | tail -3
!cd {REPO} && python scripts/cek_fogbench_tak_berubah.py

In [ ]:
# 2. definisi satu_fold / jalankan_loso (kp2/satu_fold.py)
from kp2.satu_fold import satu_fold, buat_jalankan_loso
jalankan_loso = buat_jalankan_loso(recs)

In [ ]:
# N1: muat runner dan buat struktur folder
exec(open(f"{REPO}/kp2/w4_runner.py").read())
siapkan_folder()

In [ ]:
# N2: uji kecil (Li 1 seed), periksa keluarannya sebelum menjalankan antrean penuh
jalankan_antrean(jalankan_loso, pilih=["li_1seed"])

In [ ]:
# SESI 1
jalankan_antrean(jalankan_loso, pilih=["li_1seed", "li_3seed", "stgcn_pinggul_1seed", "stgcn_pinggul_3seed"])

In [ ]:
# SESI 2
jalankan_antrean(jalankan_loso, pilih=["stgcn_skala_1seed", "stgcn_skala_3seed"])

In [ ]:
# SESI 3
jalankan_antrean(jalankan_loso, pilih=["stgcn_piksel_1seed", "stgcn_piksel_3seed"])

In [ ]:
# N4: ringkasan waktu dan analisis (membaca berkas fold di HASIL)
ringkas_waktu()
exec(open(f"{REPO}/analisis/analisis_w4.py").read())
exec(open(f"{REPO}/analisis/eda_w4.py").read())

In [ ]:
# Keluaran 2, 3, 4 (tanpa melatih ulang)
exec(open(f"{REPO}/analisis/keluaran_w4.py").read())

In [ ]:
# Diagnostik ambang ST-GCN (2 fold, seed 0; BUKAN hasil resmi)
exec(open(f"{REPO}/analisis/cek_ambang_w4.py").read())